# A2-L1 practical: Build a bronze loader and prove it keeps every delivery

Platform: Databricks Free Edition.

Run this in Databricks Free Edition. It builds two small deliveries of the same wave in memory, loads both into bronze, and shows them side by side. In production the rows come from `spark.read.option("header", True).csv(path)`.


In [ ]:
import uuid

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

CATALOG = spark.catalog.currentCatalog()
SCHEMA = f"{CATALOG}.learning_a2"
TABLE = f"{SCHEMA}.nat_raw"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

In [ ]:
def add_audit_columns(df: DataFrame, ingestion_id: str, source_file: str, delivery_version: int) -> DataFrame:
    """Add the four bronze audit columns."""
    return df.withColumns({
        "ingestion_id": F.lit(ingestion_id),
        "source_file": F.lit(source_file),
        "ingestion_timestamp": F.current_timestamp(),
        "delivery_version": F.lit(delivery_version),
    })


def ingest_rows(spark: SparkSession, rows: list[tuple[str, str]], table: str, source: str, version: int) -> int:
    """Append a delivery given as rows. Stands in for a CSV read in this demo."""
    raw = spark.createDataFrame(rows, "respondent_id STRING, final_weight STRING")
    add_audit_columns(raw, str(uuid.uuid4()), source, version).write.format("delta").mode("append").saveAsTable(table)
    metrics = DeltaTable.forName(spark, table).history(1).collect()[0]["operationMetrics"]
    return int(metrics["numOutputRows"])

In [ ]:
print(ingest_rows(spark, [("00123", "1.5"), ("00124", "")], TABLE, "w14_v1.csv", 1))
print(ingest_rows(spark, [("00123", "1.6"), ("00124", "0.9")], TABLE, "w14_v2.csv", 2))
display(spark.table(TABLE).orderBy("respondent_id", "delivery_version"))

Both deliveries are there. The leading zeros survive, and the empty weight is kept for the contract check to report.